# 01 — Master Calendar Factory

**Purpose:** Build the absolute temporal reference grid for the entire study.

**Outputs:**
- `data/analysis/windows/master_calendar_1h.parquet`
- `data/analysis/windows/window_metadata.json`
- `data/analysis/reports/calendar_qa.json`

**Logic:**
- Full window = Bybit klines coverage
- Core window = intersection of Bybit klines, funding, OI (conservative)
- Strict `[start, end_excl)` convention throughout

In [4]:
# ── Setup ──
import sys; sys.path.insert(0, "..")
from config import CFG, REPORTS_DIR
CFG.ensure_dirs()

import pandas as pd
import numpy as np
import json

print(f"Project root : {CFG.ROOT}")
print(f"Target window: [{CFG.START}, {CFG.END_EXCL})")

Project root : .
Target window: [2021-03-15 00:00:00+00:00, 2025-12-01 00:00:00+00:00)


In [6]:
# ── 1. Read bounds from each Bybit series ──

def get_date_bounds(path, name):
    """Read only the date column, return (min, max) aligned to the hour."""
    dates = pd.to_datetime(pd.read_parquet(path, columns=["date"])["date"], utc=True)
    lo, hi = dates.min().floor("h"), dates.max().floor("h")
    print(f"  {name:20s}  {lo}  →  {hi}  ({len(dates):,} rows)")
    return lo, hi

print("Series bounds:")
bounds = {}
for name, path in [
    ("bybit_klines",  CFG.FILES.bybit_klines),
    ("bybit_funding", CFG.FILES.bybit_funding),
    ("bybit_oi",      CFG.FILES.bybit_oi),
]:
    bounds[name] = get_date_bounds(path, name)

Series bounds:
  bybit_klines          2021-03-15 00:00:00+00:00  →  2025-11-30 23:00:00+00:00  (41,328 rows)
  bybit_funding         2021-03-15 00:00:00+00:00  →  2025-11-30 23:00:00+00:00  (41,328 rows)
  bybit_oi              2021-03-15 00:00:00+00:00  →  2025-11-30 23:00:00+00:00  (41,328 rows)


In [8]:
# ── 2. Compute windows ──

# Full window: anchored to klines (broadest Bybit series)
full_start = bounds["bybit_klines"][0]
full_end   = bounds["bybit_klines"][1] + pd.Timedelta(hours=1)

# Core window: strict intersection of all three series
core_start = max(b[0] for b in bounds.values())
core_end   = min(b[1] for b in bounds.values()) + pd.Timedelta(hours=1)

print(f"FULL window: [{full_start}, {full_end})")
print(f"CORE window: [{core_start}, {core_end})")

# Sanity: core must be inside full
assert core_start >= full_start
assert core_end   <= full_end

FULL window: [2021-03-15 00:00:00+00:00, 2025-12-01 00:00:00+00:00)
CORE window: [2021-03-15 00:00:00+00:00, 2025-12-01 00:00:00+00:00)


In [10]:
# ── 3. Build master calendar ──

calendar = pd.DataFrame({
    "date": pd.date_range(full_start, full_end, freq="1h", tz="UTC", inclusive="left")
})

n_expected = int((full_end - full_start) / pd.Timedelta(hours=1))
n_actual   = len(calendar)

print(f"Calendar rows: {n_actual:,}  (expected: {n_expected:,})")
assert n_actual == n_expected, f"Mismatch! {n_actual} != {n_expected}"

# QA checks
diffs = calendar["date"].diff().dropna()
assert diffs.eq(pd.Timedelta(hours=1)).all(), "Non-uniform spacing detected!"
assert calendar["date"].is_monotonic_increasing, "Not monotonic!"
assert str(calendar["date"].dt.tz) == "UTC", "Not UTC!"

print("✅ All QA checks passed")

Calendar rows: 41,328  (expected: 41,328)
✅ All QA checks passed


In [12]:
# ── 4. Save artifacts ──

# Calendar
calendar.to_parquet(CFG.FILES.master_calendar, index=False, engine="pyarrow")
print(f"Saved: {CFG.FILES.master_calendar}")

# Window metadata
metadata = {
    "full_window": {
        "start": full_start.isoformat(),
        "end_excl": full_end.isoformat(),
        "n_hours": n_actual,
    },
    "core_window": {
        "start": core_start.isoformat(),
        "end_excl": core_end.isoformat(),
        "n_hours": int((core_end - core_start) / pd.Timedelta(hours=1)),
    },
    "series_bounds": {k: {"start": v[0].isoformat(), "end": v[1].isoformat()} for k, v in bounds.items()},
    "convention": "timestamp = bucket_start, window = [start, end_excl)",
}

with open(CFG.FILES.window_metadata, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Saved: {CFG.FILES.window_metadata}")

# QA report
qa = {
    "n_hours": n_actual,
    "expected": n_expected,
    "monotonic": True,
    "uniform_1h": True,
    "timezone": "UTC",
    "status": "PASS",
}
from config import REPORTS_DIR
qa_path = REPORTS_DIR / "calendar_qa.json"
with open(qa_path, "w") as f:
    json.dump(qa, f, indent=2)
print(f"Saved: {qa_path}")

print("\n✅ Notebook 01 complete")

Saved: ./data/analysis/windows/master_calendar_1h.parquet
Saved: ./data/analysis/windows/window_metadata.json
Saved: ./data/analysis/reports/calendar_qa.json

✅ Notebook 01 complete
